# Relazione sul Progetto di Intelligenza Artificiale del gruppo Sgrogongo Gongo AI

## Addestramento di un Agente Autonomo per la Guida su Pista con Deep Q-Network (DQN)


**Membri del Gruppo:** Emiliano Chiarini, Luca Di Mauro, Lorenzo Davalli, Lorenzo Pirchio  

---

## 1. Introduzione

### 1.1 Obiettivo del Progetto

Il progetto ha come obiettivo principale lo sviluppo di un **agente intelligente** capace di apprendere autonomamente a guidare un kart su un circuito di gara, utilizzando tecniche di **Reinforcement Learning (RL)**. L'agente deve imparare a:

- Rimanere sulla pista evitando di uscire fuori pista
- Attraversare tutti i checkpoint in ordine sequenziale
- Completare il giro nel minor tempo possibile
- Ottimizzare la traiettoria per massimizzare la ricompensa cumulativa

### 1.2 Tecnologie Utilizzate

| Componente | Librerie |
|------------|------------|
| Framework RL | Gymnasium (OpenAI Gym) |
| Deep Learning | PyTorch |
| Rendering | Pygame |
| Analisi Dati | NumPy, Pandas, Matplotlib, Seaborn |
| Generazione Report | FPDF |
| Gestione Progetto | UV (Python) |

---

## 2. Background Teorico

### 2.1 Reinforcement Learning

Il **Reinforcement Learning** è un paradigma di apprendimento automatico in cui un agente impara a prendere decisioni interagendo con un ambiente. L'agente:

1. **Osserva** lo stato corrente dell'ambiente ($s_t$)
2. **Seleziona** un'azione ($a_t$) secondo una policy $\pi$
3. **Esegue** l'azione e riceve una ricompensa ($r_t$)
4. **Transita** in un nuovo stato ($s_{t+1}$)

L'obiettivo è trovare la policy ottimale $\pi^*$ che massimizza la ricompensa cumulativa attesa.

### 2.2 Deep Q-Learning

L'utilizzo delle reti neurali, che nel contesto del Q-Learning prendono il nome di Deep Q Network (DQN), cambia completamente la struttura dell’algoritmo del Q-Learning. Per comprendere al meglio come funziona il Deep Q-Learning, dividiamo l’algoritmo in fasi:

1. La prima cosa da fare è creare due DQN: la Policy Network e la Target Network. La Policy Network, o Training Network, è la rete sulla quale viene effettuato l’addestramento ed è aggiornata attivamente durante la fase di apprendimento; viene utilizzata per predire i Q-values per lo stato corrente quando l’agente ha scelto un’azione. La Target Network è la rete utilizzata per calcolare i target Q-values  per lo stato successivo durante la fase di apprendimento. Questa rete viene aggiornata meno frequentemente per aumentare la stabilità dell’apprendimento. Il motivo per cui nel Deep Q-Learning si utilizzano due DQN diverse è che esse permettono all’algoritmo di convergere più rapidamente e in modo più stabile. Per comprendere meglio il ruolo di ciascuna rete, osserviamo i passaggi seguenti.

2. Dopo aver creato le due reti, il passo successivo è copiare la Policy Network nella Target Network. Questo passaggio è necessario per garantire coesione tra le due reti all’inizio dell’algoritmo.

3. A questo punto, l’agente sceglie ed esegue l’azione successiva.

4. Inserire lo stato corrente come input nella Policy Network. Si noti che all’inizio dell’addestramento i Q-values in uscita dalla rete saranno numeri casuali, poiché le reti neurali vengono inizializzate con valori casuali per pesi e bias.

5. Inserire lo stato corrente come input nella Target Network per ottenere i Q-values in uscita.

6. Calcolare il Q-value per lo stato corrente e per l’azione scelta dall’agente. Per calcolare il Q-value, se lo stato successivo non è uno stato terminale, viene utilizzata l’equazione di Bellman; altrimenti, il Q-value è uguale alla ricompensa associata a quell’azione.

7. Il Q-value calcolato viene impostato come Q-value dell’azione scelta dall’agente nel livello di output della Target Network.

8. A questo punto, i Q-values della Target Network vengono utilizzati per addestrare la Policy Network. L’addestramento delle reti neurali utilizza la tecnica chiamata backpropagation. Questo processo fa sì che la Policy Network modifichi pesi e bias per avvicinarsi gradualmente ai Q-values target.

9. Ripetere i punti da 3 a 8 fino a quando non viene raggiunto uno stato terminale e l’addestramento termina.

10. Periodicamente, dopo un numero arbitrario di iterazioni, la Target Network viene sincronizzata con la Policy Network. Questo passaggio è cruciale perché consente di evitare instabilità e divergenza durante l’apprendimento. Esistono diversi modi per sincronizzare le due reti, ma il più intuitivo e utilizzato consiste nel copiare la Policy Network nella Target Network.

### 2.3 Processo di Decisione di Markov (MDP)

L'ambiente è formalizzato come un **Markov Decision Process** definito dalla tupla $(S, A, P, R, \gamma)$:

- $S$: Spazio degli stati (matrice di osservazione locale)
- $A$: Spazio delle azioni (4 direzioni cardinali)
- $P$: Funzione di transizione probabilistica
- $R$: Funzione di ricompensa
- $\gamma$: Fattore di sconto

---

## 3. Struttura del Progetto

```
ProgettoAI/
├── src/
│   ├── main.py                    # Entry point principale
│   ├── main_costants.py           # Costanti globali del training
│   ├── agents/
│   │   ├── agent.py               # Classi Agent e ReplayBuffer
│   │   ├── agent_costants.py      # Iperparametri dell'agente
│   │   └── network.py             # Architettura della Rete Neurale
│   ├── env/
│   │   ├── track_env.py           # Ambiente Gymnasium Custom
│   │   ├── track_costants.py      # Costanti dell'ambiente
│   │   └── track_utils.py         # Utility per il tracciato
│   └── utils/
│       ├── __init__.py            # Device detection (CPU/GPU)
│       ├── visual.py              # Generazione grafici
│       └── report_generator.py    # Generazione report PDF
├── data/
│   ├── tracks/                    # File CSV dei tracciati
│   └── music/                     # Audio usato durante il rendering
├── models/
│   └── checkpoints/               # Modelli salvati (.pth), creati durante il training
├── results/                       # Output (grafici, heatmap, report)
├── notebooks/                     # Jupyter notebooks, deprecati
└── docs/                          # Documentazione
```

---

## 4. L'Ambiente: TrackEnv

### 4.1 Descrizione Generale

L'ambiente `TrackEnv` è implementato come una sottoclasse di `gymnasium.Env`, seguendo lo standard OpenAI Gym. Questo permette la compatibilità con qualsiasi algoritmo di RL standard.

### 4.2 Rappresentazione del Circuito

Il circuito è memorizzato come una **matrice 2D** caricata da un file CSV. Ogni cella della matrice ha un valore che rappresenta il tipo di terreno:

| Valore | Costante | Significato |
|--------|----------|-------------|
| 0.0 | `TRACK_OFFROAD_VALUE` | Fuori pista (erba/muro) |
| 0.1 | `TRACK_CURB_VALUE` | Cordolo |
| 0.2 | `TRACK_ROAD_VALUE` | Strada |
| 0.3 | `TRACK_FINISH_VALUE` | Linea del traguardo |
| -0.1 | `TRACK_UNKNOWN_VALUE` | Zona non visibile |

### 4.3 Spazio delle Osservazioni

L'agente osserva una **finestra locale** della pista centrata sulla sua posizione. Questa è una matrice quadrata di dimensione `VIEW_SIZE × VIEW_SIZE` (default: 16x16 = 256 celle).  
Inoltre, nella modalità di movimento realistica (`velocity`), l'osservazione include anche il vettore della **velocità attuale** del kart.

```python
observation_space = gym.spaces.Dict({
    "agent_view": gym.spaces.Box(
        low=TRACK_UNKNOWN_VALUE, 
        high=TRACK_FINISH_VALUE,
        shape=(VIEW_SIZE, VIEW_SIZE),
        dtype=np.float32
    ),
    "speed": gym.spaces.Box( # Presente solo in modalità 'velocity'
        low=-VELOCITY_MAX_SPEED,
        high=VELOCITY_MAX_SPEED,
        shape=(2,),
        dtype=np.int32
    )
})
```

Questa osservazione locale permette all'agente di avere una **percezione limitata** della pista, simile a una vista in prima persona.

### 4.4 Spazio delle Azioni

L'ambiente supporta due modalità di movimento, configurabili tramite la costante `MOVEMENT_MODE`:

#### Modalità "velocity" (Accelerazione)
In questa modalità più realistica, l'agente controlla l'**accelerazione** del kart anziché la posizione assoluta. La velocità viene cumulata ad ogni step e la nuova posizione è calcolata in base alla velocità risultante.  
L'agente può scegliere tra **9 azioni discrete** che rappresentano le combinazioni di accelerazione sugli assi:

| Azione | Accelerazione (Riga, Colonna) | Descrizione |
|--------|-------------------------------|-------------|
| 0      | (-1, -1)                      | Alto-Sinistra |
| 1      | (-1, 0)                       | Alto |
| 2      | (-1, 1)                       | Alto-Destra |
| 3      | (0, -1)                       | Sinistra |
| 4      | (0, 0)                        | Nessuna Accelerazione |
| 5      | (0, 1)                        | Destra |
| 6      | (1, -1)                       | Basso-Sinistra |
| 7      | (1, 0)                        | Basso |
| 8      | (1, 1)                        | Basso-Destra |

```python
action_space = gym.spaces.Discrete(9) # VELOCITY_ACTION_SPACE_SIZE
```
Questa modalità include anche controlli per verificare se la traiettoria percorsa interseca zone fuori pista, rendendo la simulazione più fisica e prevenendo comportamenti non permessi ("salti" sopra l'erba).

#### Modalità "simple" (Movimento Discreto)
Nella modalità classica, l'agente sceglie semplicemente di spostarsi di una singola cella nelle **4 direzioni cardinali**:

| Azione | Direzione | Vettore Movimento |
|--------|-----------|-------------------|
| 0 | Destra | (0, +1) |
| 1 | Alto | (-1, 0) |
| 2 | Sinistra | (0, -1) |
| 3 | Basso | (+1, 0) |

```python
action_space = gym.spaces.Discrete(4)
```


### 4.5 Checkpoints

Il sistema di ricompensa dell'ambiente si basa sulla presenza di una serie di checkpoint lungo il circuito.
Questi checkpoint non sono altro che liste di celle della matrice circuito.

Ogni checkpoint è costruito in funzione della sua distanza dalla linea di partenza, infatti le celle appartenenti ad un checkpoint sono tutte quelle celle che hanno la stessa distanza dal traguardo.
Tutti i checkpoint sono costruiti in modo dinamico alla matrice del circuito.

Infatti, dato un numero arbitrario di checkpoint che si vuole avere nella pista, vengono prima definite le distanze a cui i checkpoint devono essere dal traguardo per essere distribuiti in modo equidistante gli uni dagl'altri. In seguito vengo creati i checkpoint in sé, delle liste in cui vengono inserite tutte le celle  della matrice circuito che hanno la distanza pari alla distanza definita per quel determinato checkpoint.

Nella fase di training, quando l'agente passa per un checkpoint, viene ricompensato in modo molto positivo, però per evitare che l'agente cerchi di compiere comportamenti e strategie 'illegali', come andare direttamente al traguardo senza passare per i checkpoint, è stato implementato un sistema di progresso.

Infatti, il traguardo e i checkpoint non sono sempre attivi, ma sono attivati sequenzialmente uno dopo l'altro. Per esempio il terzo checkpoint non sarà attivo se l'agente non ha attraversato il secondo, e così via. Il traguardo è attivo soltanto quando l'agente ha attraversato tutti i checkpoint.

Ovviamente il progresso dell'agente è viene resettato ad ogni episodio.

### 4.6 Sistema di Ricompense

La funzione di reward è usata per guidare l'apprendimento dell'agente. E' stata progettata per:

- **Incentivare** il progresso verso il traguardo
- **Penalizzare** comportamenti indesiderati
- **Evitare** loop infiniti o stalli

#### Tabella dei Reward

| Evento | Valore | Descrizione |
|--------|--------|-------------|
| `FINISH_REWARD` | +100 | Completamento giro (tutti i checkpoint superati) |
| `CHECKPOINT_REWARD` | +30 | Attraversamento checkpoint |
| `OFFROAD_REWARD` | -50 | Uscita di pista (termina episodio) |
| `STEP_PENALTY` | -0.01 | Penalità per ogni step (incentiva velocità) |
| `REPEAT_PENALTY` | -0.5 | Penalità per visitare celle già visitate |
| `BACKWARD_PENALTY` | ×1 | Moltiplicatore per movimento all'indietro |
| Progresso positivo | +delta | Ricompensa proporzionale all'avanzamento |
| Prima visita cella | +0.1 | Bonus per esplorazione |

### 4.7 Rendering con Pygame

L'ambiente include un sistema di rendering visivo in stile **Mario Kart** che permette di visualizzare:

- **Il circuito** con erba, strada, cordoli e traguardo
- **Il kart** con animazioni di rotazione e ombra
- **I checkpoint** rappresentati come stelle animate
- **Effetti di drifting** (fumo) quando l'agente cambia direzione
- **Decorazioni** come funghi e banane, generati in modo procedurale ogni volta
- **HUD** con progressi checkpoint e tempo

---

## 5. L'Agente: Deep Q-Network

### 5.1 Architettura della Rete Neurale

La rete neurale `Network` è un **Multi-Layer Perceptron (MLP)** che prende in input l'osservazione appiattita e produce i Q-values per ogni azione.

#### Implementazione PyTorch

```python
class Network(nn.Module):
    def __init__(self, input_dim, action_size, seed=42):
        super(Network, self).__init__()
        self.seed = torch.manual_seed(seed)
        
        self.fc1 = nn.Linear(input_dim, 256)   # 256 → 256
        self.fc2 = nn.Linear(256, 256)         # 256 → 256
        self.fc3 = nn.Linear(256, action_size) # 256 → 4

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # Q-values
```

### 5.2 Experience Replay

Un aspetto fondamentale dell'addestramento di una rete neurale è quello di addestrarla con dati di addestramento randomizzati.

Questo è molto importante perché elimina la correlazione temporale tra i campioni, consentendo alla rete neurale di generalizzare più facilmente e di non apprendere una situazione molto specifica.

Per fare questo, nel Deep Q Learning, viene utilizzato l'experience replay. L'experience replay consiste nell'utilizzare un array per memorizzare la memoria dell'agente. Questa memoria ha una lunghezza fissa e ogni volta che si riempie, l'ultimo elemento, ovvero la memoria più vecchia, viene eliminato dall'array.

Quando l'agente sceglie un'azione e la compie (fase 3), invece di passare lo stato direttamente alla Policy Network, memorizziamo i dati di questa interazione tra l'agente e l'ambiente (la tupla stato, azione, stato successivo, ricompensa ) nella memoria.

Successivamente, un numero arbitrario di memorie viene campionato in modo casuale dalla memoria e passato come dati di addestramento per la Policy Network (fase 4).

A seguito viene riportata la definizione della classe `ReplayBuffer`, la quale implementa la memoria dell'experience replay:


```python
class ReplayBuffer:
    def __init__(self, capacity, batch_size, seed=42):
        self.memory = deque(maxlen=capacity)
        self.batch_size = batch_size
    
    def add(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    
    def sample(self):
        batch = random.sample(self.memory, self.batch_size)
        return zip(*batch)
```

### 5.3 Iperparametri dell'Agente

| Parametro | Valore | Descrizione |
|-----------|--------|-------------|
| `LEARNING_RATE` | 0.0001 | Tasso di apprendimento |
| `MINIBATCH_SIZE` | 100 | Dimensione batch per training |
| `REPLAY_BUFFER_SIZE` | 200,000 | Capacità memoria replay |
| `DISCOUNT_FACTOR` (γ) | 0.99 | Peso ricompense future |
| `EPSILON_START` | 1.0 | Epsilon iniziale |
| `EPSILON_MIN` | 0.01 | Epsilon minimo finale |
| `EPSILON_DECAY_PORTION` | 0.8 | Porzione episodi per decay |
| `TARGET_UPDATE_EVERY` | 4 | Frequenza aggiornamento della rete |

### 5.4 Strategia Epsilon-Greedy

L'esplorazione è gestita tramite una strategia **epsilon-greedy** con decadimento. Sono supportati due tipi di decay:

1. Decadimento Lineare
2. Decadimento Esponenziale (Default)


```python
def update_epsilon(self):
    match eps_type:
        case "lineare": 
            self.epsilon = max(epsilon_min, epsilon - decay)
        case "esponenziale": 
            self.epsilon = max(epsilon_min, epsilon * decay)
```

### 5.5 Algoritmo di Apprendimento

La funzione `learn()` implementa l'aggiornamento dei pesi della Q-network usando la **Temporal Difference (TD) loss**:

```python
def learn(self):
    # Campiona mini-batch dal replay buffer
    states, actions, rewards, next_states, dones = self.memory.sample()
    
    # Calcola Q-values previsti
    q_expected = self.q_net(states).gather(1, actions)
    
    # Calcola Q-target usando la target network
    with torch.no_grad():
        q_next = self.target_net(next_states).max(1)[0]
        q_target = rewards + (gamma * q_next * (1 - dones))
    
    # Calcola loss MSE e aggiorna pesi
    loss = F.mse_loss(q_expected, q_target)
    
    self.optimizer.zero_grad()
    loss.backward()
    self.optimizer.step()
```

---

## 6. Pipeline di Training

### 6.1 Flusso di Esecuzione  
Il progetto viene lanciato col il comando:

```bash
python src/main.py train
```

C'é la possibilità di variare il numero di episodi usati per il training col l'apposita estensione ``` --ep NUM_EPISODI ```


1. **Inizializzazione**
   - Crea l'ambiente `TrackEnv`
   - Inizializza l'agente con Q-network e target network
   - Crea il replay buffer vuoto

2. **Loop degli Episodi**
   ```python
   for episodio in 1..DEFAULT_TRAIN_EPISODES:
       state = env.reset()
       
       while not done:
           action = agent.select_action(state)
           next_state, reward, done = env.step(action)
           agent.step(state, action, reward, next_state, done)
           state = next_state
       
       agent.update_epsilon()  # Decadimento di epsilon
       
       if is_best_average:
           save_model("best_model.pth")
       
       if episodio % SAVE_CHECKPOINT_EVERY == 0:
           save_checkpoint(f"cp_{episodio}.pth")
   ```

3. **Output Finale**
   - Grafico andamento reward
   - Heatmap delle traiettorie
   - Report PDF

### 6.2 Parametri di Training

| Parametro | Valore | Descrizione |
|-----------|--------|-------------|
| `DEFAULT_TRAIN_EPISODES` | 1000 | Numero episodi default |
| `STEP_LIMIT` | 1000 | Step massimi per episodio |
| `SCORES_WINDOW_SIZE` | 1000 | Finestra media mobile |
| `SAVE_CHECKPOINT_EVERY` | 1000 | Frequenza salvataggio |

---

## 7. Testing e Valutazione

### 7.1 Modalità Test

```bash
python src/main.py test
```

C'é la possibilità di cambiare il modello usato col l'apposita estensione ``` --mod cp_XXXX.pth ```

In modalità test:
- Epsilon è impostato a **0** (non esplora, ma fa quello che sa)
- Il rendering è attivo per visualizzare l'agente

### 7.2 Metriche di Valutazione

1. **Score Medio**: Media mobile dei reward totali per episodio
2. **Heatmap Traiettorie**: Visualizza le celle visitate con scala logaritmica
3. **Completamento Giro**: Percentuale di episodi che raggiungono il traguardo
4. **Tempo per Checkpoint**: Step medi per raggiungere ogni checkpoint

---

## 8. Circuiti Disponibili

### 8.1 Track Imola

Rappresenta il Circuito di Imola (ma dove poi?). È rappresentato come una matrice CSV con le seguenti caratteristiche:

- Dimensioni: matrice 60×60
- Larghezza pista: 7 celle
- Checkpoint: 7

### 8.2 Track Ovale

Un circuito (quasi) ovale per test e debug.

### 8.3 Generazione Matrice Distanze

La matrice delle distanze è sostanzialmente una matrice "ombra" della matrice rappresentante il circuito, dove i valori delle celle, al posto di rappresentare le differenti parti del circuito, indicano la distanza di ogni cella dal punto di partenza. La generazione della matrice delle distanze avviene tramite un semplice algoritmo di propagazione.  
Innanzitutto viene inizializzata la matrice delle distanze come una copia della matrice rappresentante il circuito. In seguito viene assegnato il valore -2 a tutte le celle non appartenenti all'insieme delle celle "fuori-strada", valore che indica che alla cella non è stato assegnato il valore della distanza dal traguardo a quella cella.  
L'algoritmo continua, definendo l'insieme di celle che saranno utilizzate come il punto iniziale del calcolo delle distanze e che definiscono la "direzione di propagazione" dell'algoritmo. Questo insieme è definito manualmente dallo sviluppatore indicando tra quale delle 4 direzioni ("su", "giù", "destra", "sinistra") rispetto alla linea di partenza la propagazione deve iniziare. Per esempio, se la direzione "destra" è stata utilizzata, le celle immediatamente a destra della linea di partenza, saranno le prime celle dell'algoritmo di calcolo delle distanze.  
Un ultima operazione che viene fatta prima di calcolare le distanze è, ovviamente, assegnare alle celle della linea di partenza la distanza 0.  
Ora vediamo come funziona il sistema di calcolo delle distanze.  
Viene inizializzato una lista contenente l'insieme delle celle immediatamente dopo la linea di partenza citate prima.  
Iterativamente dalla lista, finché questa non è vuota, viene estratto il primo elemento secondo la logica FIFO. Una volta estratto un elemento, per ognuno delle quattro celle immediatamente adiacenti, viene calcolata la distanza come la distanza della cella estratta dalla lista + 1.  
Il valore della cella vicina viene cambiato solo se tre condizioni sono verificate:

1. gli indici della cella vicina esistono nella matrice
2. il valore di quella cella nella matrice delle distanze è -2 e quindi non ancora è ancora stata calcolata la distanza per quella cella
3. la cella non fa parte del traguardo  

Se tutte e tre le condizioni sono vere, allora la distanza è calcolata e la cella vicina è aggiunta in fondo alla lista.  
Questo algoritmo fa in modo che il calcolo della distanza si propaghi in ogni direzione del tracciato.


---

## 9. Strumenti di Analisi

### 9.1 Grafico Training

La funzione `save_training_plot()` genera un grafico PNG dell'andamento del training:

- **Asse X**: Numero di episodio
- **Asse Y**: Reward totale (score)
- **Curva**: Media mobile su 50 episodi

### 9.2 Heatmap

Il sistema genera due tipi di heatmap:

1. **heatmap.csv**: Matrice con conteggio visite per cella
2. **heatmap.png**: Visualizzazione con scala logaritmica colorata

La scala logaritmica permette di visualizzare sia celle poco visitate che aree ad alta densità.

### 9.3 Report PDF

Al termine del training viene generato un report PDF contenente:

1. **Parametri Agente**: Tutti gli iperparametri usati
2. **Parametri Pista**: Configurazione dell'ambiente
3. **Architettura Rete**: Layer e dimensioni
4. **Grafico Training**: Andamento dei reward
5. **Heatmap**: Traiettorie percorse
6. **Codice Sorgente**: Funzione step() completa

---

## 10. Ottimizzazioni

1. **Reward Shaping Avanzato**
   - Bonus per esplorazione di nuove celle
   - Penalità crescente per celle rivisitate
   - Terminazione episodio i8n caso di oscillazione del modello

2. **Sistema Checkpoint**
   - Impedisce le scorciatoie
   - Fornisce reward intermedi per progresso

3. **Supporto GPU** --> alla fine funziona?
   - Rilevamento automatico CUDA
   - Trasferimento tensori su device appropriato


---

## 11. Conclusioni

ah boh, funziona

---
### 13 Repository

Il codice sorgente completo è disponibile nella repository GitHub del progetto.